<a href="https://colab.research.google.com/github/erDcs10/Automated-Essay-Scoring--AES--Bahasa-Indonesia/blob/main/notebook/data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Hasil Pekerjaan Kelompok 11


*   Erlangga Deanda Chandra Setya (244311013)
*   Muhammad Rofiqul Anam (244311019)

## MINGGU 2 — [L] DATA UNDERSTANDING & DATA PREPARATION

**Tujuan:** Data bersih, siap dipakai, dan terbagi dengan benar tanpa kebocoran informasi (leakage).

**Tugas:**
1. Isi informasi dataset (jumlah baris/kolom, tipe, ukuran, format) dan tabel deskripsi fitur (Bab 4.1–4.2).
2. Analisis kondisi data: missing values, duplikat, outlier, imbalance, noise (Bab 4.3).
3. EDA (opsional): minimal 3 visualisasi beserta insight (Bab 4.4).
4. Data cleaning dan feature engineering (Bab 5.1–5.2).
5. Transformasi data sesuai jenis data (encoding, scaling, tokenisasi, augmentasi, dll.) (Bab 5.3).
6. **Splitting** train/validation/test dengan `random_state` tetap; gunakan stratified jika klasifikasi (Bab 5.4).
7. Data balancing jika diperlukan, **hanya pada data training** (Bab 5.5).
8. Tulis ringkasan Apa–Mengapa–Bagaimana (Bab 5.6).
9. Commit notebook `01_learning_problem_eda` dan `02_data_preparation` ke GitHub.

**Deliverable:**
- Bab 4–5 selesai
- Notebook data preparation yang bisa dijalankan ulang

**Checklist selesai:**
- [✔️] Scaler/encoder di-fit hanya pada data training
- [✔️] Test set disimpan dan tidak disentuh sampai Minggu 6
- [✔️] Jumlah sampel tiap split tercatat
- [ ] Setiap langkah preprocessing punya alasan tertulis


#DOWNLOAD

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("realdeo/ukara-enhanced")

print("Path to dataset files:", path)

100%|██████████| 237M/237M [00:03<00:00, 65.8MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/realdeo/ukara-enhanced/versions/5


#LOAD


In [2]:
import os
import pandas as pd

# 1. Load dataset dari path kagglehub
dataset_dir = "/root/.cache/kagglehub/datasets/realdeo/ukara-enhanced/versions/5"
print("File tersedia:", os.listdir(dataset_dir))

# Sesuaikan nama file CSV jika berbeda (umumnya dataset.csv)
csv_path = os.path.join(dataset_dir, "dataset.csv")
df = pd.read_csv(csv_path)

# 2. Inspeksi struktur & kolom
print(f"\nShape awal: {df.shape}")
print("Nama kolom:", df.columns.tolist())
print("\nMissing values per kolom:\n", df.isnull().sum())
print("\n5 Baris pertama:\n", df.head())

File tersedia: ['idwiki_word2vec.model.trainables.syn1neg.npy', 'idwiki_word2vec.model', 'stopword_list.txt', 'idwiki_word2vec.model.wv.vectors.npy', 'dataset.csv']

Shape awal: (12215, 3)
Nama kolom: ['kelompok', 'teks', 'label']

Missing values per kolom:
 kelompok    0
teks        0
label       0
dtype: int64

5 Baris pertama:
   kelompok                                               teks  label
0        1  Mahasiswa harus membuat daftar batu mana yang ...      0
1        1  Anda akan memerlukan lebih banyak potongan-pot...      1
2        1  Beberapa informasi tambahan yang akan anda but...      1
3        1  Tambahan informasi yang akan saya butuhkan dal...      0
4        1  Tambahan informasi yang akan anda butuhkan unt...      1


## CLEANING

In [3]:
import os
import re
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [4]:
# ==========================================
# DATA CLEANING (BAB 5.1)
# ==========================================
df_clean = df.dropna(subset=['teks', 'label', 'kelompok']).copy()
df_clean['teks'] = df_clean['teks'].astype(str)
df_clean['kelompok'] = df_clean['kelompok'].astype(str)
df_clean['label'] = pd.to_numeric(df_clean['label'], errors='coerce')
df_clean = df_clean.dropna(subset=['label'])
df_clean['label'] = df_clean['label'].astype(int)

# Hapus duplikat berdasarkan pasangan kelompok soal dan teks jawaban
df_clean = df_clean.drop_duplicates(subset=['kelompok', 'teks']).reset_index(drop=True)

# Kamus normalisasi slang / singkatan umum Indonesia
slang_dict = {
    'yg': 'yang', 'dgn': 'dengan', 'sdh': 'sudah', 'udah': 'sudah',
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'tdk': 'tidak', 'nggak': 'tidak',
    'utk': 'untuk', 'krn': 'karena', 'karna': 'karena', 'dr': 'dari',
    'dlm': 'dalam', 'bgt': 'sangat', 'bs': 'bisa', 'jg': 'juga',
    'tp': 'tetapi', 'tapi': 'tetapi', 'klo': 'kalau', 'kl': 'kalau',
    'org': 'orang', 'jd': 'jadi', 'sy': 'saya', 'aku': 'saya',
    'td': 'tadi', 'blm': 'belum', 'sm': 'sama', 'pd': 'pada'
}

def clean_indonesian_essay(text):
    text = text.lower()
    text = re.sub(r'<.*?>|http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s.,?!]', ' ', text)
    text = re.sub(r'([.,?!])', r' \1 ', text)
    words = text.split()
    normalized_words = [slang_dict.get(w, w) for w in words]
    clean_text = ' '.join(normalized_words)
    clean_text = re.sub(r'\s+([.,?!])', r'\1', clean_text)
    return re.sub(r'\s+', ' ', clean_text).strip()

df_clean['clean_teks'] = df_clean['teks'].apply(clean_indonesian_essay)
df_clean = df_clean[df_clean['clean_teks'].str.len() > 0].reset_index(drop=True)

# Tambahkan konteks kelompok soal ke input teks untuk IndoBERT
df_clean['indobert_input'] = "kelompok " + df_clean['kelompok'] + " : " + df_clean['clean_teks']

print(f"Shape setelah cleaning: {df_clean.shape}")
print("\nDistribusi Kelas Target (label):")
print(df_clean['label'].value_counts())

Shape setelah cleaning: (12173, 5)

Distribusi Kelas Target (label):
label
0    7630
1    4543
Name: count, dtype: int64


#SPLITTING

In [5]:
# ==========================================
# DATA SPLITTING 80/10/10 (BAB 5.4)
# ==========================================
strat_col = df_clean['kelompok'] + "_" + df_clean['label'].astype(str)
if strat_col.value_counts().min() < 5:
    strat_col = df_clean['label']

train_df, temp_df = train_test_split(
    df_clean, test_size=0.20, random_state=42, stratify=strat_col
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print(f"\nTrain set : {len(train_df)} baris")
print(f"Val set   : {len(val_df)} baris")
print(f"Test set  : {len(test_df)} baris")


Train set : 9738 baris
Val set   : 1217 baris
Test set  : 1218 baris


#TRANSFORM

In [6]:
import os

stopword_path = os.path.join(dataset_dir, "stopword_list.txt")
print(f"Apakah file stopword ada? {os.path.exists(stopword_path)}")

if os.path.exists(stopword_path):
    with open(stopword_path, 'r', encoding='utf-8') as f:
        stopwords = [next(f).strip() for _ in range(20)]
    print("\n20 stopword pertama:")
    print(stopwords)

Apakah file stopword ada? True

20 stopword pertama:
['ada', 'adalah', 'adanya', 'adapun', 'agak', 'agaknya', 'agar', 'akan', 'akankah', 'akhir', 'akhiri', 'akhirnya', 'aku', 'akulah', 'amat', 'amatlah', 'anda', 'andalah', 'antar', 'antara']


In [7]:
import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load stopword_list.txt bawaan dari direktori dataset UKARA Enhanced
stopword_path = os.path.join(dataset_dir, "stopword_list.txt")
with open(stopword_path, "r", encoding="utf-8") as f:
  indonesian_stopwords = [line.strip() for line in f if line.strip()]

print(f"Jumlah stopwords dimuat: {len(indonesian_stopwords)} kata")


# 2. Ekstraksi fitur struktural teks (Bab 5.2) pada masing-masing split
def add_text_features(df_split):
  df_feat = df_split.copy()
  df_feat["word_count"] = df_feat["clean_teks"].apply(lambda x: len(x.split()))
  df_feat["char_count"] = df_feat["clean_teks"].apply(
      lambda x: len(x.replace(" ", ""))
  )
  df_feat["avg_word_length"] = df_feat["char_count"] / (
      df_feat["word_count"] + 1e-5
  )
  return df_feat


train_ml_df = add_text_features(train_df)
val_ml_df = add_text_features(val_df)
test_ml_df = add_text_features(test_df)

Jumlah stopwords dimuat: 758 kata


In [8]:
# ==========================================
# TRANSFORMASI JALUR A: TF-ID, OneHotEncoder, StandardScaler (BAB 5.3)
# ==========================================
# Definisi kolom fitur
text_feature = "clean_teks"
cat_features = ["kelompok"]
num_features = ["word_count", "char_count", "avg_word_length"]

# Susun ColumnTransformer
preprocessor_jalur_a = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=5000,
                stop_words=indonesian_stopwords,
                sublinear_tf=True,
            ),
            text_feature,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("num", StandardScaler(), num_features),
    ],
    remainder="drop",
)

# FIT HANYA PADA DATA TRAINING (mencegah data leakage!)
X_train_ml = preprocessor_jalur_a.fit_transform(train_ml_df)
X_val_ml = preprocessor_jalur_a.transform(val_ml_df)
X_test_ml = preprocessor_jalur_a.transform(test_ml_df)

# Target label (0 dan 1)
y_train_ml = train_ml_df["label"].values
y_val_ml = val_ml_df["label"].values
y_test_ml = test_ml_df["label"].values

print("\nVerifikasi Dimensi Matriks Fitur Jalur A:")
print(f"X_train_ml shape : {X_train_ml.shape}")
print(f"X_val_ml shape   : {X_val_ml.shape}")
print(f"X_test_ml shape  : {X_test_ml.shape}")

/usr/local/lib/python3.13/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(



Verifikasi Dimensi Matriks Fitur Jalur A:
X_train_ml shape : (9738, 5014)
X_val_ml shape   : (1217, 5014)
X_test_ml shape  : (1218, 5014)


In [9]:
# ==========================================
# TRANSFORMASI JALUR B: INDOBERT (BAB 5.3)
# ==========================================
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Cek panjang token maksimum di train_df
token_lengths = [len(tokenizer.encode(t, truncation=False)) for t in train_df['indobert_input']]
print(f"\nPanjang token rata-rata: {sum(token_lengths)/len(token_lengths):.1f} | Maksimum: {max(token_lengths)}")

MAX_LEN = 128  # Jika maksimum > 128, naikkan ke 256

class UkaraClassificationDataset(Dataset):
    def __init__(self, dataframe, text_col, label_col, tokenizer, max_len):
        self.texts = dataframe[text_col].tolist()
        self.labels = dataframe[label_col].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)  # torch.long untuk klasifikasi
        }

BATCH_SIZE = 16
train_dataset = UkaraClassificationDataset(train_df, 'indobert_input', 'label', tokenizer, MAX_LEN)
val_dataset   = UkaraClassificationDataset(val_df, 'indobert_input', 'label', tokenizer, MAX_LEN)
test_dataset  = UkaraClassificationDataset(test_df, 'indobert_input', 'label', tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

sample = next(iter(train_loader))
print("\nVerifikasi Tensor Batch Jalur B:")
print("input_ids shape     :", sample['input_ids'].shape)
print("attention_mask shape:", sample['attention_mask'].shape)
print("labels shape        :", sample['labels'].shape)

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Panjang token rata-rata: 46.3 | Maksimum: 335

Verifikasi Tensor Batch Jalur B:
input_ids shape     : torch.Size([16, 128])
attention_mask shape: torch.Size([16, 128])
labels shape        : torch.Size([16])


#BALANCING

In [13]:
import numpy as np
import pandas as pd
import torch
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils.class_weight import compute_class_weight

# ==========================================================
# 1. CEK DISTRIBUSI KELAS PADA TRAINING SET
# ==========================================================
class_counts = train_df['label'].value_counts()
class_ratio = train_df['label'].value_counts(normalize=True) * 100

print('Distribusi Kelas pada Train Set (Sebelum Balancing):')
for cls in class_counts.index:
  print(f'Kelas {cls}: {class_counts[cls]} sampel ({class_ratio[cls]:.2f}%)')

# Ambang batas (threshold) imbalance: jika kelas minoritas < 40%
is_imbalanced = class_ratio.min() < 10.0
print(
    f'\nStatus Imbalance: {"PERLU BALANCING" if is_imbalanced else "CUKUP SEIMBANG"}'
)

# ==========================================================
# 2. BALANCING JALUR A (HANYA PADA X_train_ml & y_train_ml)
# ==========================================================
if is_imbalanced:
  ros_a = RandomOverSampler(random_state=42)
  X_train_ml_bal, y_train_ml_bal = ros_a.fit_resample(X_train_ml, y_train_ml)
  print('\n[Jalur A] Shape X_train_ml setelah RandomOverSampler:')
  print(
      f'X_train_ml_bal: {X_train_ml_bal.shape} | Distribusi y:'
      f' {np.bincount(y_train_ml_bal)}'
  )
else:
  X_train_ml_bal, y_train_ml_bal = X_train_ml, y_train_ml

# ==========================================================
# 3. BALANCING JALUR B: INDOBERT (HANYA PADA train_df)
# ==========================================================
# Pendekatan 1 (Direkomendasikan untuk NLP): Menghitung Class Weights untuk Loss Function
classes = np.unique(train_df['label'])
weights = compute_class_weight(
    class_weight='balanced', classes=classes, y=train_df['label']
)
class_weights_tensor = torch.tensor(weights, dtype=torch.float)
print(
    f'\n[Jalur B - Opsi 1] Class Weights untuk CrossEntropyLoss:'
    f' {class_weights_tensor.tolist()}'
)

# Pendekatan 2 (Alternatif): Random Oversampling baris teks pada train_df saja
if is_imbalanced:
  ros_b = RandomOverSampler(random_state=42)
  train_df_bal, _ = ros_b.fit_resample(train_df, train_df['label'])
  train_df_bal = train_df_bal.reset_index(drop=True)
  print(
      f'[Jalur B - Opsi 2] Jumlah baris train_df_bal:'
      f' {len(train_df_bal)} (Val & Test tetap utuh!)'
  )
else:
  train_df_bal = train_df.copy()

Distribusi Kelas pada Train Set (Sebelum Balancing):
Kelas 0: 6104 sampel (62.68%)
Kelas 1: 3634 sampel (37.32%)

Status Imbalance: CUKUP SEIMBANG

[Jalur B - Opsi 1] Class Weights untuk CrossEntropyLoss: [0.7976736426353455, 1.339845895767212]


#MODEL

In [14]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, f1_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

# ==========================================================
# MODEL 1: BASELINE (LOGISTIC REGRESSION) - BAB 6.1
# ==========================================================
start_time = time.time()
model_baseline = LogisticRegression(C=1.0, max_iter=500, random_state=42)
model_baseline.fit(X_train_ml_bal, y_train_ml_bal)
train_time_m1 = time.time() - start_time

# Evaluasi pada Validation Set (distribusi asli)
y_val_pred_m1 = model_baseline.predict(X_val_ml)
print("=== HASIL VALIDASI MODEL 1: LOGISTIC REGRESSION (BASELINE) ===")
print(f"Training Time : {train_time_m1:.2f} detik")
print(f"Accuracy      : {accuracy_score(y_val_ml, y_val_pred_m1):.4f}")
print(f"F1-Score (Mac): {f1_score(y_val_ml, y_val_pred_m1, average='macro'):.4f}")
print(f"Cohen's Kappa : {cohen_kappa_score(y_val_ml, y_val_pred_m1):.4f}\n")

# ==========================================================
# MODEL 2: ADVANCED ML (XGBOOST + TUNING) - BAB 6.2
# ==========================================================
start_time = time.time()
xgb_base = XGBClassifier(
    random_state=42, eval_metric="logloss", n_jobs=-1
)

# Hyperparameter Tuning sederhana (wajib di Bab 6.2.4)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6],
    "learning_rate": [0.1, 0.2],
}

grid_xgb = GridSearchCV(
    xgb_base, param_grid, cv=3, scoring="f1_macro", n_jobs=-1
)
grid_xgb.fit(X_train_ml_bal, y_train_ml_bal)
model_advanced = grid_xgb.best_estimator_
train_time_m2 = time.time() - start_time

# Evaluasi pada Validation Set (distribusi asli)
y_val_pred_m2 = model_advanced.predict(X_val_ml)
print("=== HASIL VALIDASI MODEL 2: XGBOOST (ADVANCED ML) ===")
print(f"Best Params   : {grid_xgb.best_params_}")
print(f"Training Time : {train_time_m2:.2f} detik")
print(f"Accuracy      : {accuracy_score(y_val_ml, y_val_pred_m2):.4f}")
print(f"F1-Score (Mac): {f1_score(y_val_ml, y_val_pred_m2, average='macro'):.4f}")
print(f"Cohen's Kappa : {cohen_kappa_score(y_val_ml, y_val_pred_m2):.4f}")

=== HASIL VALIDASI MODEL 1: LOGISTIC REGRESSION (BASELINE) ===
Training Time : 0.45 detik
Accuracy      : 0.8743
F1-Score (Mac): 0.8658
Cohen's Kappa : 0.7316

=== HASIL VALIDASI MODEL 2: XGBOOST (ADVANCED ML) ===
Best Params   : {'learning_rate': 0.2, 'max_depth': 6, 'n_estimators': 200}
Training Time : 179.07 detik
Accuracy      : 0.8710
F1-Score (Mac): 0.8614
Cohen's Kappa : 0.7228


In [15]:
import time
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# ==========================================================
# MODEL 3: DEEP LEARNING FINE-TUNED INDOBERT - BAB 6.2
# ==========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# Load IndoBERT untuk Klasifikasi Biner (num_labels=2)
model_dl = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=2,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
).to(device)

# Gunakan Class Weights dari Opsi 1 untuk menangani imbalance
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = torch.optim.AdamW(model_dl.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 10  # Sesuai syarat minimum >= 10 epochs di Bab 3.4
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}
best_val_f1 = 0.0

start_dl = time.time()
for epoch in range(EPOCHS):
    # --- TRAINING ---
    model_dl.train()
    total_train_loss, correct_train, total_train = 0, 0, 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_dl(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_dl.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    # --- VALIDATION ---
    model_dl.eval()
    total_val_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model_dl(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    train_acc = correct_train / total_train
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model_dl.state_dict(), "best_indobert_ukara.pt")

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

print(f"\nTotal Waktu Training IndoBERT: {(time.time() - start_dl)/60:.2f} menit | Best Val F1: {best_val_f1:.4f}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Menggunakan device: cuda


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/10 | Train Loss: 0.4640 | Val Loss: 0.3427 | Val Acc: 0.8439 | Val F1: 0.8331
Epoch 02/10 | Train Loss: 0.3272 | Val Loss: 0.3294 | Val Acc: 0.8718 | Val F1: 0.8669
Epoch 03/10 | Train Loss: 0.2569 | Val Loss: 0.3062 | Val Acc: 0.8792 | Val F1: 0.8735
Epoch 04/10 | Train Loss: 0.2125 | Val Loss: 0.4089 | Val Acc: 0.8652 | Val F1: 0.8609
Epoch 05/10 | Train Loss: 0.1830 | Val Loss: 0.3764 | Val Acc: 0.8817 | Val F1: 0.8771
Epoch 06/10 | Train Loss: 0.1530 | Val Loss: 0.4799 | Val Acc: 0.8784 | Val F1: 0.8729
Epoch 07/10 | Train Loss: 0.1301 | Val Loss: 0.5011 | Val Acc: 0.8825 | Val F1: 0.8754


KeyboardInterrupt: 